# DrawRNN — Quick, Draw! Sketch Classifier

Interactive notebook for exploring the dataset, training the model, and visualizing results.

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import from src/

import random
import torch
import torch.nn as nn

from src.config import Config
from src.dataset import build_label_maps, load_category, process_drawing, create_dataloaders
from src.model import DrawRNNModel
from src.train import train, evaluate
from src.visualize import draw_sketch

## 1. Configuration

In [ ]:
cfg = Config()
encode, decode = build_label_maps(cfg.categories)
print(f"Categories: {cfg.categories}")
print(f"Encode: {encode}")
print(f"Decode: {decode}")

## 2. Load & Explore Data

In [ ]:
raw = {}
for cat in cfg.categories:
    raw[cat] = load_category(cfg.data_dir, cat)
    print(f"{cat}: {len(raw[cat])} drawings")

In [ ]:
# Visualize a random drawing
cat = random.choice(cfg.categories)
idx = random.randint(0, len(raw[cat]) - 1)
draw_sketch(raw[cat][idx], delay=0.05, animated=False)

## 3. Process Data

In [ ]:
all_data = []
for cat in cfg.categories:
    for d in raw[cat]:
        all_data.append(process_drawing(d, encode[cat]))

random.shuffle(all_data)
print(f"Total processed drawings: {len(all_data)}")
print(f"Example shape: {all_data[0]['coordinates'].shape}")

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    all_data, batch_size=cfg.batch_size,
    train_ratio=cfg.train_ratio, val_ratio=cfg.val_ratio,
)
print(f"Train: {len(train_loader.dataset)}")
print(f"Val:   {len(val_loader.dataset)}")
print(f"Test:  {len(test_loader.dataset)}")

## 4. Build & Train Model

In [ ]:
model = DrawRNNModel(
    input_size=cfg.input_size,
    hidden_size=cfg.hidden_size,
    output_size=cfg.num_classes,
    num_layers=cfg.num_layers,
    dropout=cfg.dropout,
)
print(model)
print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)

model = train(
    model, train_loader, val_loader, criterion, optimizer,
    num_epochs=cfg.num_epochs, device=cfg.device,
    patience=cfg.patience, min_delta=cfg.min_delta,
    checkpoint_dir=cfg.checkpoint_dir,
)

## 5. Evaluate on Test Set

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, cfg.device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")